# 00 - Bootstrap the Fabric Lakehouse

Creates the control-plane, attempt-output, reconciliation, benchmark, and gold Delta tables plus committed-output views. Run once per environment and again only for reviewed, backward-compatible schema releases.

**Before running:** In the Lakehouse explorer, attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timezone
import re

from delta.tables import DeltaTable
from pyspark.sql import SparkSession


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def identifier(value: str, name: str, *, allow_empty: bool = False) -> str:
    if allow_empty and not value:
        return ""
    if IDENTIFIER.fullmatch(value) is None:
        raise ValueError(f"{name} is not a valid SQL identifier: {value!r}")
    return value


database = identifier(DATABASE.strip(), "DATABASE", allow_empty=True)
prefix = identifier(TABLE_PREFIX.strip(), "TABLE_PREFIX")
spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("Attach a Fabric Lakehouse and start a Spark session")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")

if database:
    create_database_sql = "CREATE DATABASE IF NOT EXISTS `" + database + "`"
    spark_session.sql(create_database_sql)


def name(suffix: str) -> str:
    table = f"{prefix}_{suffix}"
    return f"`{database}`.`{table}`" if database else f"`{table}`"


def storage_name(suffix: str) -> str:
    table = f"{prefix}_{suffix}"
    return f"{database}.{table}" if database else table


ddl = {
    "event_receipts": """
        event_key STRING NOT NULL,
        event_source STRING NOT NULL,
        event_id STRING NOT NULL,
        event_type STRING NOT NULL,
        event_time TIMESTAMP NOT NULL,
        subject STRING NOT NULL,
        manifest_uri STRING NOT NULL,
        work_id STRING,
        received_at TIMESTAMP NOT NULL,
        registration_status STRING NOT NULL,
        pipeline_run_id STRING,
        error_type STRING,
        error_message STRING
    """,
    "video_work": """
        work_id STRING NOT NULL,
        asset_id STRING NOT NULL,
        asset_version STRING NOT NULL,
        source_uri STRING NOT NULL,
        manifest_uri STRING NOT NULL,
        source_etag STRING NOT NULL,
        expected_size_bytes BIGINT NOT NULL,
        expected_sha256 STRING,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        camera_timezone STRING NOT NULL,
        duration_seconds DOUBLE,
        priority INT NOT NULL,
        status STRING NOT NULL,
        received_at TIMESTAMP NOT NULL,
        queued_at TIMESTAMP NOT NULL,
        not_before_at TIMESTAMP,
        attempt_count INT NOT NULL,
        max_attempts INT NOT NULL,
        lease_owner_attempt_id STRING,
        lease_dispatcher_id STRING,
        lease_acquired_at TIMESTAMP,
        lease_expires_at TIMESTAMP,
        last_heartbeat_at TIMESTAMP,
        committed_attempt_id STRING,
        completed_at TIMESTAMP,
        last_error_category STRING,
        last_error_type STRING,
        last_error_message STRING,
        config_json STRING NOT NULL,
        config_sha256 STRING NOT NULL,
        capture_date DATE NOT NULL
    """,
    "video_attempts": """
        attempt_id STRING NOT NULL,
        work_id STRING NOT NULL,
        dispatcher_id STRING NOT NULL,
        pipeline_run_id STRING,
        activity_run_id STRING,
        fabric_job_instance_id STRING,
        sdk_version STRING,
        bundle_manifest_sha256 STRING,
        config_sha256 STRING NOT NULL,
        status STRING NOT NULL,
        claimed_at TIMESTAMP NOT NULL,
        staging_started_at TIMESTAMP,
        inference_started_at TIMESTAMP,
        writing_started_at TIMESTAMP,
        completed_at TIMESTAMP,
        last_heartbeat_at TIMESTAMP,
        input_sha256 STRING,
        source_size_bytes BIGINT,
        source_duration_seconds DOUBLE,
        source_fps DOUBLE,
        total_source_frames BIGINT,
        processed_frames BIGINT,
        effective_sample_fps DOUBLE,
        processing_seconds DOUBLE,
        distinct_people BIGINT,
        line_in_count BIGINT,
        line_out_count BIGINT,
        retryable BOOLEAN,
        error_category STRING,
        error_type STRING,
        error_message STRING,
        capture_date DATE NOT NULL
    """,
    "dispatcher_leases": """
        lock_name STRING NOT NULL,
        owner_id STRING NOT NULL,
        acquired_at TIMESTAMP NOT NULL,
        expires_at TIMESTAMP NOT NULL
    """,
    "registration_leases": """
        lock_name STRING NOT NULL,
        owner_id STRING NOT NULL,
        acquired_at TIMESTAMP NOT NULL,
        expires_at TIMESTAMP NOT NULL
    """,
    "replay_requests": """
        replay_id STRING NOT NULL,
        work_id STRING NOT NULL,
        requested_by STRING NOT NULL,
        reason STRING NOT NULL,
        requested_at TIMESTAMP NOT NULL,
        previous_status STRING NOT NULL,
        applied_at TIMESTAMP,
        capture_date DATE NOT NULL
    """,
    "telemetry_attempts": """
        work_id STRING NOT NULL,
        attempt_id STRING NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        capture_date DATE NOT NULL,
        recorded_at TIMESTAMP NOT NULL,
        person_id BIGINT NOT NULL,
        entry_frame BIGINT NOT NULL,
        exit_frame BIGINT NOT NULL,
        entry_seconds DOUBLE NOT NULL,
        exit_seconds DOUBLE NOT NULL,
        person_entry_at_utc TIMESTAMP NOT NULL,
        person_exit_at_utc TIMESTAMP NOT NULL,
        duration_seconds DOUBLE NOT NULL
    """,
    "line_count_attempts": """
        work_id STRING NOT NULL,
        attempt_id STRING NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        capture_date DATE NOT NULL,
        recorded_at TIMESTAMP NOT NULL,
        frame BIGINT NOT NULL,
        video_seconds STRING NOT NULL,
        video_timestamp STRING NOT NULL,
        observed_at_utc TIMESTAMP NOT NULL,
        frame_in_count BIGINT NOT NULL,
        frame_out_count BIGINT NOT NULL,
        cumulative_in_count BIGINT NOT NULL,
        cumulative_out_count BIGINT NOT NULL,
        line_start_x BIGINT NOT NULL,
        line_start_y BIGINT NOT NULL,
        line_end_x BIGINT NOT NULL,
        line_end_y BIGINT NOT NULL
    """,
    "reconciliation_findings": """
        finding_id STRING NOT NULL,
        detected_at TIMESTAMP NOT NULL,
        last_detected_at TIMESTAMP NOT NULL,
        severity STRING NOT NULL,
        finding_type STRING NOT NULL,
        work_id STRING,
        attempt_id STRING,
        details STRING NOT NULL,
        resolved_at TIMESTAMP,
        capture_date DATE
    """,
    "processing_benchmarks": """
        benchmark_id STRING NOT NULL,
        benchmark_batch_id STRING NOT NULL,
        benchmark_started_at TIMESTAMP NOT NULL,
        completed_at TIMESTAMP NOT NULL,
        capacity_sku STRING NOT NULL,
        runtime_version STRING NOT NULL,
        sdk_version STRING NOT NULL,
        config_sha256 STRING NOT NULL,
        sample_name STRING NOT NULL,
        video_duration_seconds DOUBLE NOT NULL,
        end_to_end_seconds DOUBLE NOT NULL,
        overhead_seconds DOUBLE NOT NULL,
        processing_seconds DOUBLE NOT NULL,
        speed_x_realtime DOUBLE NOT NULL,
        peak_memory_mb DOUBLE,
        concurrent_workers INT NOT NULL,
        succeeded BOOLEAN NOT NULL,
        error_message STRING
    """,
    "gold_flow_minute": """
        minute_utc TIMESTAMP NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        entries BIGINT NOT NULL,
        exits BIGINT NOT NULL,
        net_flow BIGINT NOT NULL,
        source_videos BIGINT NOT NULL,
        refreshed_at TIMESTAMP NOT NULL,
        flow_date DATE NOT NULL
    """,
    "gold_flow_hour": """
        hour_utc TIMESTAMP NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        entries BIGINT NOT NULL,
        exits BIGINT NOT NULL,
        net_flow BIGINT NOT NULL,
        source_videos BIGINT NOT NULL,
        refreshed_at TIMESTAMP NOT NULL,
        flow_date DATE NOT NULL
    """,
    "gold_video": """
        work_id STRING NOT NULL,
        captured_at_utc TIMESTAMP NOT NULL,
        camera_id STRING NOT NULL,
        location_id STRING NOT NULL,
        video_duration_seconds DOUBLE,
        processing_seconds DOUBLE,
        speed_x_realtime DOUBLE,
        distinct_people BIGINT,
        line_in_count BIGINT,
        line_out_count BIGINT,
        completed_at TIMESTAMP NOT NULL,
        capture_date DATE NOT NULL
    """,
    "gold_operations_hour": """
        hour_utc TIMESTAMP NOT NULL,
        queued BIGINT NOT NULL,
        started BIGINT NOT NULL,
        succeeded BIGINT NOT NULL,
        failed BIGINT NOT NULL,
        video_hours_completed DOUBLE NOT NULL,
        average_processing_seconds DOUBLE,
        p95_processing_seconds DOUBLE,
        refreshed_at TIMESTAMP NOT NULL,
        operation_date DATE NOT NULL
    """
}

partitioned = {
    "video_work": "capture_date",
    "video_attempts": "capture_date",
    "replay_requests": "capture_date",
    "telemetry_attempts": "capture_date",
    "line_count_attempts": "capture_date",
    "gold_flow_minute": "flow_date",
    "gold_flow_hour": "flow_date",
    "gold_video": "capture_date",
    "gold_operations_hour": "operation_date",
}

for suffix, columns in ddl.items():
    # Fabric Runtime's Spark SQL parser does not accept column-level
    # NOT NULL constraints in this CREATE TABLE USING DELTA form.
    compatible_columns = columns.replace(" NOT NULL", "")
    empty = spark_session.createDataFrame([], schema=compatible_columns)
    writer = empty.write.format("delta").mode("ignore")
    if suffix in partitioned:
        writer = writer.partitionBy(partitioned[suffix])
    writer.saveAsTable(storage_name(suffix))

In [ ]:
epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
registration_seed = spark_session.createDataFrame(
    [("global", "", epoch, epoch)],
    schema=spark_session.table(storage_name("registration_leases")).schema,
)
(
    DeltaTable.forName(spark_session, storage_name("registration_leases"))
    .alias("t")
    .merge(registration_seed.alias("s"), "t.lock_name = s.lock_name")
    .whenNotMatchedInsertAll()
    .execute()
)

telemetry_view_sql = (
    "CREATE OR REPLACE VIEW " + name("telemetry_committed") + " AS "
    "SELECT t.* FROM " + name("telemetry_attempts") + " t "
    "INNER JOIN " + name("video_work") + " w "
    "ON t.work_id = w.work_id "
    "AND t.attempt_id = w.committed_attempt_id "
    "WHERE w.status = 'SUCCEEDED'"
)
spark_session.sql(telemetry_view_sql)

line_counts_view_sql = (
    "CREATE OR REPLACE VIEW " + name("line_counts_committed") + " AS "
    "SELECT l.* FROM " + name("line_count_attempts") + " l "
    "INNER JOIN " + name("video_work") + " w "
    "ON l.work_id = w.work_id "
    "AND l.attempt_id = w.committed_attempt_id "
    "WHERE w.status = 'SUCCEEDED'"
)
spark_session.sql(line_counts_view_sql)

runs_view_sql = (
    "CREATE OR REPLACE VIEW " + name("runs_committed") + " AS "
    "SELECT w.work_id, w.asset_id, w.asset_version, w.camera_id, "
    "w.location_id, w.captured_at_utc, w.camera_timezone, "
    "w.duration_seconds, w.committed_attempt_id, w.completed_at, "
    "w.config_sha256, a.processing_seconds, a.effective_sample_fps, "
    "a.processed_frames, a.distinct_people, a.line_in_count, "
    "a.line_out_count, a.input_sha256 FROM " + name("video_work") + " w "
    "INNER JOIN " + name("video_attempts") + " a "
    "ON w.work_id = a.work_id "
    "AND w.committed_attempt_id = a.attempt_id "
    "WHERE w.status = 'SUCCEEDED'"
)
spark_session.sql(runs_view_sql)

expected = {f"{prefix}_{suffix}" for suffix in ddl}
show_tables = "SHOW TABLES IN `" + database + "`" if database else "SHOW TABLES"
actual = {row.tableName for row in spark_session.sql(show_tables).collect()}
missing = sorted(expected - actual)
if missing:
    raise RuntimeError(f"Bootstrap did not create tables: {missing}")

print(f"Created or verified {len(expected)} Delta tables and 3 committed views")